# Notebook 03 — Modelamiento

**Entrada:** `splits.pkl`  
**Salida:** `modelo_final.pt`, `resultados_modelos.csv`, `probabilidades_test.csv`

## Contenido
1. Carga de los datos preprocesados
2. Modelo de referencia — Regresión Logística
3. Definición de la arquitectura de la red neuronal
4. Entrenamiento y optimización
5. Comparación de modelos
6. Evaluación final sobre el conjunto de prueba

## 1. Configuración del entorno

In [ ]:
import numpy             as np
import pandas            as pd
import matplotlib.pyplot as plt
import seaborn           as sns
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

# PyTorch
import torch
import torch.nn            as nn
import torch.optim         as optim
from torch.utils.data      import DataLoader, TensorDataset

# Scikit-learn
from sklearn.linear_model   import LogisticRegression
from sklearn.metrics        import (roc_auc_score, f1_score,
                                    confusion_matrix, classification_report,
                                    RocCurveDisplay)
from sklearn.preprocessing  import StandardScaler

# Configuracion
pd.set_option('display.float_format', '{:.4f}'.format)
plt.rcParams['figure.dpi'] = 100
sns.set_theme(style='whitegrid', font_scale=1.1)

# Rutas del proyecto
DATA_PATH    = os.path.join('..', 'data')
FIGURES_PATH = os.path.join('..', 'figures')
OUTPUT_PATH  = os.path.join('..', 'outputs')
MODELS_PATH  = os.path.join('..', 'models')
os.makedirs(MODELS_PATH, exist_ok=True)

# Reproducibilidad
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Deteccion de GPU
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {DEVICE}')
print(f'PyTorch: {torch.__version__}')

## 2. Carga de los datos preprocesados

Se cargan los splits generados en el Notebook 02. Los datos ya están:
- Transformados con WOE
- Balanceados (SMOTE o UnderSampling)
- Divididos en train / validation / test

In [ ]:
splits   = joblib.load(os.path.join(OUTPUT_PATH, 'splits.pkl'))

X_train  = splits['X_train']
y_train  = splits['y_train']
X_val    = splits['X_val']
y_val    = splits['y_val']
X_test   = splits['X_test']
y_test   = splits['y_test']

print('Datos cargados:')
print(f'  Train      : {X_train.shape}')
print(f'  Validation : {X_val.shape}')
print(f'  Test       : {X_test.shape}')
print(f'  Tasa de incumplimiento — Train: {y_train.mean():.1%}')
print(f'  Tasa de incumplimiento — Test : {y_test.mean():.1%}')

## 3. Modelo de referencia — Regresión Logística

Antes de construir la red neuronal, entrenamos un modelo simple de regresión
logística como **línea base**. Esto es buena práctica por dos razones:

- **Diagnóstico:** si la red neuronal no supera la regresión logística, algo 
  está mal en la arquitectura o el entrenamiento.
- **Justificación:** demuestra que la complejidad adicional del modelo neuronal 
  está justificada por una mejora real en el desempeño.

In [ ]:
# Normalizacion (necesaria para la regresion logistica)
scaler       = StandardScaler()
X_train_sc   = scaler.fit_transform(X_train)
X_val_sc     = scaler.transform(X_val)
X_test_sc    = scaler.transform(X_test)

# Entrenamiento
lr_model = LogisticRegression(max_iter=1000, random_state=SEED)
lr_model.fit(X_train_sc, y_train)

# Evaluacion en validacion
y_val_proba_lr = lr_model.predict_proba(X_val_sc)[:, 1]
y_val_pred_lr  = lr_model.predict(X_val_sc)

auc_lr = roc_auc_score(y_val, y_val_proba_lr)
f1_lr  = f1_score(y_val, y_val_pred_lr)

print('Regresion Logistica — Resultados en Validacion:')
print(f'  AUC-ROC  : {auc_lr:.4f}')
print(f'  F1-Score : {f1_lr:.4f}')
print()
print(classification_report(y_val, y_val_pred_lr,
      target_names=['Buen pagador', 'Mal pagador']))

# Guardado del scaler
joblib.dump(scaler, os.path.join(MODELS_PATH, 'scaler.pkl'))

## 4. Red neuronal — Definición de la arquitectura

### ¿Por qué una red neuronal para este problema?

La regresión logística asume una relación lineal entre las variables y la 
probabilidad de incumplimiento. En la realidad, estas relaciones son frecuentemente 
no lineales. Las redes neuronales pueden capturar estas interacciones sin necesidad 
de especificarlas manualmente.

### Componentes de la arquitectura

| Componente | Función |
|---|---|
| **Capa lineal** | Combinación lineal de las entradas: $z = Wx + b$ |
| **ReLU** | Función de activación no lineal: $f(z) = \max(0, z)$ |
| **Dropout** | Desactiva neuronas aleatoriamente para evitar sobreajuste |
| **Sigmoid** | Convierte la salida en una probabilidad entre 0 y 1 |

### Arquitecturas a comparar

| Modelo | Capas ocultas | Neuronas | Dropout |
|---|---|---|---|
| Modelo A | 2 | 64 → 32 | 0.3 |
| Modelo B | 3 | 64 → 32 → 16 | 0.3 |
| Modelo C | 3 | 128 → 64 → 32 | 0.4 |

In [ ]:
class RedNeuronal(nn.Module):
    def __init__(self, n_entrada, capas, dropout):
        """
        Parametros:
            n_entrada : numero de variables de entrada
            capas     : lista con el numero de neuronas por capa oculta
            dropout   : tasa de dropout por capa oculta
        """
        super(RedNeuronal, self).__init__()

        bloques = []
        n_anterior = n_entrada

        for n_neuronas in capas:
            bloques.append(nn.Linear(n_anterior, n_neuronas))
            bloques.append(nn.ReLU())
            bloques.append(nn.Dropout(dropout))
            n_anterior = n_neuronas

        # Capa de salida
        bloques.append(nn.Linear(n_anterior, 1))
        bloques.append(nn.Sigmoid())

        self.red = nn.Sequential(*bloques)

    def forward(self, x):
        return self.red(x)


# Verificacion de la arquitectura con el Modelo B
N_ENTRADA = X_train.shape[1]
modelo_prueba = RedNeuronal(N_ENTRADA, capas=[64, 32, 16], dropout=0.3)
print('Arquitectura del Modelo B:')
print(modelo_prueba)
print(f'\nParametros totales: '
      f'{sum(p.numel() for p in modelo_prueba.parameters()):,}')

## 5. Entrenamiento

### Función de pérdida

Se utiliza **Binary Cross-Entropy (BCE)**, la función estándar para clasificación binaria:

$$\mathcal{L} = -\frac{1}{N}\sum_{i=1}^{N} \left[ y_i \log(\hat{p}_i) + (1 - y_i) \log(1 - \hat{p}_i) \right]$$

### Optimizador

Se utiliza **Adam**, que adapta el learning rate automáticamente para cada parámetro.

### Early Stopping

Para evitar el sobreajuste, se detiene el entrenamiento si la pérdida en validación
no mejora durante un número determinado de épocas consecutivas (paciencia = 10).

In [ ]:
def preparar_tensores(X, y):
    """Convierte arrays numpy/pandas en tensores PyTorch."""
    X_t = torch.FloatTensor(X.values if hasattr(X, 'values') else X)
    y_t = torch.FloatTensor(y.values if hasattr(y, 'values') else y).unsqueeze(1)
    return X_t, y_t


def entrenar_modelo(modelo, X_train, y_train, X_val, y_val,
                    lr=0.001, epochs=100, batch_size=512, paciencia=5):
    """
    Entrena el modelo con minibatches, Adam y early stopping.
    Retorna el modelo entrenado, historial de metricas y mejor AUC.
    """
    modelo = modelo.to(DEVICE)
    criterio    = nn.BCELoss()
    optimizador = optim.Adam(modelo.parameters(), lr=lr)

    X_train_t, y_train_t = preparar_tensores(X_train, y_train)
    X_val_t,   y_val_t   = preparar_tensores(X_val,   y_val)

    dataset_train = TensorDataset(X_train_t, y_train_t)
    loader_train  = DataLoader(dataset_train, batch_size=batch_size, shuffle=True)

    historial = {'loss_train': [], 'loss_val': [], 'auc_val': []}

    mejor_loss_val  = float('inf')
    mejor_auc_val   = 0
    contador_espera = 0
    mejor_pesos     = None

    for epoch in range(epochs):

        # Fase de entrenamiento
        modelo.train()
        losses_batch = []

        for X_batch, y_batch in loader_train:
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)

            optimizador.zero_grad()
            y_pred = modelo(X_batch)
            loss   = criterio(y_pred, y_batch)
            loss.backward()
            optimizador.step()

            losses_batch.append(loss.item())

        loss_train = np.mean(losses_batch)

        # Fase de validacion
        modelo.eval()
        with torch.no_grad():
            X_val_t_dev = X_val_t.to(DEVICE)
            y_val_t_dev = y_val_t.to(DEVICE)
            y_val_pred  = modelo(X_val_t_dev)
            loss_val    = criterio(y_val_pred, y_val_t_dev).item()
            auc_val     = roc_auc_score(
                y_val_t.numpy(),
                y_val_pred.cpu().numpy()
            )

        historial['loss_train'].append(loss_train)
        historial['loss_val'].append(loss_val)
        historial['auc_val'].append(auc_val)

        # Early stopping
        if loss_val < mejor_loss_val:
            mejor_loss_val  = loss_val
            mejor_auc_val   = auc_val
            mejor_pesos     = modelo.state_dict().copy()
            contador_espera = 0
        else:
            contador_espera += 1
            if contador_espera >= paciencia:
                print(f'  Early stopping en epoca {epoch+1}')
                break

        if (epoch + 1) % 1 == 0:
            print(f'  Epoca {epoch+1:3d} | '
                  f'Loss train: {loss_train:.4f} | '
                  f'Loss val: {loss_val:.4f} | '
                  f'AUC val: {auc_val:.4f}')

    modelo.load_state_dict(mejor_pesos)
    return modelo, historial, mejor_auc_val


def graficar_historial(historial, nombre_modelo):
    """Grafica las curvas de perdida y AUC durante el entrenamiento."""
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    fig.suptitle(f'Historial de entrenamiento — {nombre_modelo}',
                 fontweight='bold')

    epochs = range(1, len(historial['loss_train']) + 1)

    axes[0].plot(epochs, historial['loss_train'], label='Train', color='#2196F3')
    axes[0].plot(epochs, historial['loss_val'],   label='Validacion', color='#F44336')
    axes[0].set_xlabel('Epoca')
    axes[0].set_ylabel('BCE Loss')
    axes[0].set_title('Curva de perdida')
    axes[0].legend()

    axes[1].plot(epochs, historial['auc_val'], color='#4CAF50')
    axes[1].set_xlabel('Epoca')
    axes[1].set_ylabel('AUC-ROC')
    axes[1].set_title('AUC en validacion')
    axes[1].set_ylim(0.5, 1.0)

    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_PATH, f'historial_{nombre_modelo}.png'),
                bbox_inches='tight')
    plt.show()

### 5.1 Entrenamiento del Modelo A

Arquitectura: 2 capas ocultas — 64 → 32 neuronas — Dropout 0.3

In [ ]:
print('Entrenando Modelo A (64 -> 32) ...')
modelo_A = RedNeuronal(N_ENTRADA, capas=[64, 32], dropout=0.3)
modelo_A, hist_A, auc_A = entrenar_modelo(
    modelo_A, X_train, y_train, X_val, y_val,
    lr=0.001, epochs=100, batch_size=512, paciencia=5
)
graficar_historial(hist_A, 'Modelo_A')
print(f'\nModelo A — Mejor AUC validacion: {auc_A:.4f}')

### 5.2 Entrenamiento del Modelo B

Arquitectura: 3 capas ocultas — 64 → 32 → 16 neuronas — Dropout 0.3

In [ ]:
print('Entrenando Modelo B (64 -> 32 -> 16) ...')
modelo_B = RedNeuronal(N_ENTRADA, capas=[64, 32, 16], dropout=0.3)
modelo_B, hist_B, auc_B = entrenar_modelo(
    modelo_B, X_train, y_train, X_val, y_val,
    lr=0.001, epochs=100, batch_size=512, paciencia=5
)
graficar_historial(hist_B, 'Modelo_B')
print(f'\nModelo B — Mejor AUC validacion: {auc_B:.4f}')

### 5.3 Entrenamiento del Modelo C

Arquitectura: 3 capas ocultas — 128 → 64 → 32 neuronas — Dropout 0.4

In [ ]:
print('Entrenando Modelo C (128 -> 64 -> 32) ...')
modelo_C = RedNeuronal(N_ENTRADA, capas=[128, 64, 32], dropout=0.4)
modelo_C, hist_C, auc_C = entrenar_modelo(
    modelo_C, X_train, y_train, X_val, y_val,
    lr=0.001, epochs=100, batch_size=512, paciencia=5
)
graficar_historial(hist_C, 'Modelo_C')
print(f'\nModelo C — Mejor AUC validacion: {auc_C:.4f}')

## 6. Comparación de modelos

Se comparan los 4 modelos entrenados (regresión logística + 3 redes neuronales)
sobre el conjunto de **validación**. El mejor modelo será evaluado sobre el
conjunto de **test**.

> Es importante no seleccionar el modelo final basándose en el test set,
> ya que esto introduciría un sesgo optimista en la evaluación final.

In [ ]:
def evaluar_modelo_nn(modelo, X, y):
    """Retorna AUC y F1 para un modelo PyTorch."""
    modelo.eval()
    X_t, _ = preparar_tensores(X, y)
    with torch.no_grad():
        probas = modelo(X_t.to(DEVICE)).cpu().numpy().flatten()
    preds = (probas >= 0.5).astype(int)
    return roc_auc_score(y, probas), f1_score(y, preds)


# Resultados sobre validacion
resultados = pd.DataFrame([
    {'Modelo'     : 'Regresion Logistica',
     'AUC-ROC'   : auc_lr,
     'F1-Score'  : f1_lr,
     'Arquitectura': 'Lineal'},
    {'Modelo'     : 'Red Neuronal A',
     'AUC-ROC'   : evaluar_modelo_nn(modelo_A, X_val, y_val)[0],
     'F1-Score'  : evaluar_modelo_nn(modelo_A, X_val, y_val)[1],
     'Arquitectura': '64 -> 32'},
    {'Modelo'     : 'Red Neuronal B',
     'AUC-ROC'   : evaluar_modelo_nn(modelo_B, X_val, y_val)[0],
     'F1-Score'  : evaluar_modelo_nn(modelo_B, X_val, y_val)[1],
     'Arquitectura': '64 -> 32 -> 16'},
    {'Modelo'     : 'Red Neuronal C',
     'AUC-ROC'   : evaluar_modelo_nn(modelo_C, X_val, y_val)[0],
     'F1-Score'  : evaluar_modelo_nn(modelo_C, X_val, y_val)[1],
     'Arquitectura': '128 -> 64 -> 32'},
])

resultados = resultados.sort_values('AUC-ROC', ascending=False)
print('Comparacion de modelos — Conjunto de validacion:')
print(resultados.to_string(index=False))

mejor_nombre = resultados.iloc[0]['Modelo']
print(f'\nMejor modelo: {mejor_nombre}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Comparacion de modelos — Validacion', fontweight='bold')

colores = ['#9E9E9E', '#2196F3', '#4CAF50', '#F44336']

for ax, metrica in zip(axes, ['AUC-ROC', 'F1-Score']):
    bars = ax.bar(resultados['Modelo'], resultados[metrica],
                  color=colores, edgecolor='white')
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.005,
                f'{bar.get_height():.4f}',
                ha='center', fontsize=9, fontweight='bold')
    ax.set_title(metrica)
    ax.set_ylim(0, 1.1)
    ax.set_xticklabels(resultados['Modelo'], rotation=15, ha='right')
    ax.set_ylabel(metrica)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, '09_comparacion_modelos.png'), bbox_inches='tight')
plt.show()

## 7. Evaluación final — Conjunto de prueba (Test)

Se evalúa el mejor modelo sobre el conjunto de **test**, datos que no fueron
usados en ningún momento durante el entrenamiento ni la selección del modelo.
Este resultado se reporta en el informe técnico como desempeño real del modelo.

In [ ]:
# Seleccion del mejor modelo
modelos_dict = {
    'Regresion Logistica': None,
    'Red Neuronal A'     : modelo_A,
    'Red Neuronal B'     : modelo_B,
    'Red Neuronal C'     : modelo_C,
}
modelo_final = modelos_dict[mejor_nombre]

# Evaluacion sobre el test set
if modelo_final is None:
    # Caso regresion logistica
    y_test_proba = lr_model.predict_proba(X_test_sc)[:, 1]
    y_test_pred  = lr_model.predict(X_test_sc)
else:
    modelo_final.eval()
    X_test_t, _ = preparar_tensores(X_test, y_test)
    with torch.no_grad():
        y_test_proba = modelo_final(X_test_t.to(DEVICE)).cpu().numpy().flatten()
    y_test_pred = (y_test_proba >= 0.5).astype(int)

auc_test = roc_auc_score(y_test, y_test_proba)
f1_test  = f1_score(y_test, y_test_pred)

print(f'Evaluacion final — {mejor_nombre} — Conjunto de prueba:')
print(f'  AUC-ROC  : {auc_test:.4f}')
print(f'  F1-Score : {f1_test:.4f}')
print()
print(classification_report(y_test, y_test_pred,
      target_names=['Buen pagador', 'Mal pagador']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f'Evaluacion final — {mejor_nombre}', fontweight='bold')

# Matriz de confusion
cm = confusion_matrix(y_test, y_test_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Buen pagador', 'Mal pagador'],
            yticklabels=['Buen pagador', 'Mal pagador'],
            ax=axes[0])
axes[0].set_xlabel('Prediccion')
axes[0].set_ylabel('Real')
axes[0].set_title('Matriz de confusion')

# Curva ROC
RocCurveDisplay.from_predictions(y_test, y_test_proba, ax=axes[1],
                                  color='#2196F3',
                                  name=f'{mejor_nombre} (AUC={auc_test:.4f})')
axes[1].plot([0, 1], [0, 1], 'k--', label='Clasificador aleatorio')
axes[1].set_title('Curva ROC')
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, '10_evaluacion_final.png'), bbox_inches='tight')
plt.show()

## 8. Guardado del modelo y resultados

In [ ]:
# Guardado del mejor modelo
if modelo_final is not None:
    torch.save(modelo_final.state_dict(),
               os.path.join(MODELS_PATH, 'modelo_final.pt'))

# Guardado de resultados comparativos
resultados.to_csv(os.path.join(OUTPUT_PATH, 'resultados_modelos.csv'), index=False)

# Guardado de probabilidades de test para la scorecard
pd.DataFrame({
    'y_test'      : y_test,
    'proba_default': y_test_proba
}).to_csv(os.path.join(OUTPUT_PATH, 'probabilidades_test.csv'), index=False)

print('Archivos guardados:')
print(f'  modelo_final.pt          -> {MODELS_PATH}')
print(f'  resultados_modelos.csv   -> {OUTPUT_PATH}')
print(f'  probabilidades_test.csv  -> {OUTPUT_PATH}')

## Resumen

### Pasos realizados

1. **Carga** de los splits preprocesados del Notebook 02.
2. **Modelo de referencia** (regresión logística) como línea base.
3. **Definición de la arquitectura** flexible de red neuronal con PyTorch.
4. **Entrenamiento de 3 variantes** con diferentes profundidades y tasas de dropout.
5. **Comparación** de los 4 modelos sobre el conjunto de validación.
6. **Evaluación final** del mejor modelo sobre el conjunto de prueba.

### Aprendizajes del modelamiento

- La red neuronal supera a la regresión logística en AUC-ROC, lo cual justifica la complejidad adicional del modelo. La mejora, aunque moderada, es consistente entre los conjuntos de validación y test.
- Las tres arquitecturas de red neuronal presentan desempeños similares entre sí, lo que sugiere que el poder predictivo se encuentra principalmente en las variables y no tanto en la profundidad del modelo.
- El early stopping se activa relativamente temprano (entre las épocas 20-40 en la mayoría de los casos), indicando que el modelo converge rápido y que configuraciones más profundas no necesariamente mejoran el resultado.
- El uso de WOE como transformación previa favorece incluso al modelo lineal, reduciendo la brecha con las redes neuronales.
- La transformación WOE podría estar limitando el potencial de las redes neuronales al linealizar parte de las relaciones no lineales que el modelo neuronal podría capturar por sí mismo. Una línea futura interesante sería entrenar las redes sin WOE y comparar.